# Experiment: Baidu OCR 图像识别测试

## 目标
测试百度OCR API的图像识别能力，评估其对考古报告中文字、表格、手写内容的识别效果。

## 成功标准
- 能正确获取access token
- 能调用至少3种OCR API（通用文字、表格、高精度）
- 能输出结构化的识别结果

In [ ]:
import os
import requests
import base64
import json
from pathlib import Path

print("Libraries loaded.")

## 1. 获取 Access Token

百度OCR API需要先用API Key和Secret Key换取access token。

In [ ]:
def get_access_token():
    url = "https://aip.baidubce.com/oauth/2.0/token"
    params = {
        "grant_type": "client_credentials",
        "client_id": os.environ["BAIDU_OCR_API_KEY"],
        "client_secret": os.environ["BAIDU_OCR_SECRET_KEY"],
    }
    resp = requests.post(url, params=params)
    resp.raise_for_status()
    return resp.json()["access_token"]

token = get_access_token()
print(f"Access token: {token[:20]}...")

## 2. 可用的OCR API列表

百度OCR提供多种图像识别能力，以下是对考古报告最有用的几种：

In [ ]:
OCR_APIS = {
    "general_basic": {
        "name": "通用文字识别（标准版）",
        "endpoint": "/rest/2.0/ocr/v1/general_basic",
        "use_case": "报告正文文字"
    },
    "accurate": {
        "name": "通用文字识别（高精度含位置版）",
        "endpoint": "/rest/2.0/ocr/v1/accurate",
        "use_case": "带坐标定位的精确提取"
    },
    "table": {
        "name": "表格识别",
        "endpoint": "/rest/2.0/ocr/v1/table",
        "use_case": "墓葬登记表、器物统计表"
    },
    "handwriting": {
        "name": "手写文字识别",
        "endpoint": "/rest/2.0/ocr/v1/handwriting",
        "use_case": "手写批注、田野记录"
    },
    "webimage": {
        "name": "网络图片文字识别",
        "endpoint": "/rest/2.0/ocr/v1/webimage",
        "use_case": "低质量扫描件"
    },
}

print("可用的OCR API:")
for code, info in OCR_APIS.items():
    print(f"  - {code}: {info['name']} → {info['use_case']}")

## 3. OCR调用函数

封装通用的OCR调用逻辑，支持任意图片路径。

In [ ]:
def call_ocr(token, image_path, api_type="general_basic"):
    """调用百度OCR API识别图片文字。
    
    Args:
        token: access token
        image_path: 图片文件路径
        api_type: API类型，见OCR_APIS字典
    
    Returns:
        API返回的JSON结果
    """
    if api_type not in OCR_APIS:
        raise ValueError(f"Unknown API type: {api_type}. Choose from: {list(OCR_APIS.keys())}")
    
    endpoint = OCR_APIS[api_type]["endpoint"]
    url = f"https://aip.baidubce.com{endpoint}?access_token={token}"
    
    with open(image_path, "rb") as f:
        img_base64 = base64.b64encode(f.read())
    
    resp = requests.post(url, data={"image": img_base64})
    resp.raise_for_status()
    return resp.json()

print("OCR function ready.")

## 4. 测试：通用文字识别

将图片路径替换为实际的考古报告截图或器物线图。

In [ ]:
# TODO: 替换为实际图片路径
TEST_IMAGE = "../data/raw/test_image.png"

if Path(TEST_IMAGE).exists():
    result = call_ocr(token, TEST_IMAGE, "general_basic")
    print(f"识别到 {result.get('words_result_num', 0)} 行文字：")
    for item in result.get("words_result", []):
        print(f"  {item['words']}")
else:
    print(f"图片不存在: {TEST_IMAGE}")
    print("请将测试图片放到 data/raw/ 目录下，然后修改 TEST_IMAGE 路径。")

## 5. 测试：表格识别

表格识别能提取图片中的表格结构，适合处理墓葬登记表。

In [ ]:
if Path(TEST_IMAGE).exists():
    result = call_ocr(token, TEST_IMAGE, "table")
    print(json.dumps(result, indent=2, ensure_ascii=False)[:2000])
else:
    print(f"图片不存在: {TEST_IMAGE}")

## 6. 测试：高精度识别（含位置信息）

高精度版返回每个文字框的坐标位置，可用于版面分析。

In [ ]:
if Path(TEST_IMAGE).exists():
    result = call_ocr(token, TEST_IMAGE, "accurate")
    print(f"识别到 {result.get('words_result_num', 0)} 个文字框：")
    for i, item in enumerate(result.get("words_result", [])[:5]):
        words = item["words"]
        loc = item.get("location", {})
        print(f"  [{i}] '{words}' @ ({loc.get('left', '?')}, {loc.get('top', '?')})")
    if len(result.get("words_result", [])) > 5:
        print(f"  ... 共 {len(result['words_result'])} 个")
else:
    print(f"图片不存在: {TEST_IMAGE}")

## 7. 批量处理函数

对整个目录的图片批量执行OCR，输出汇总结果。

In [ ]:
def batch_ocr(token, image_dir, api_type="general_basic", extensions=(".png", ".jpg", ".jpeg", ".bmp")):
    """批量OCR处理目录中的所有图片。
    
    Returns:
        dict: {filename: [识别到的文字行]}
    """
    image_dir = Path(image_dir)
    results = {}
    
    for img_path in sorted(image_dir.iterdir()):
        if img_path.suffix.lower() in extensions:
            print(f"Processing: {img_path.name}...", end=" ")
            try:
                resp = call_ocr(token, str(img_path), api_type)
                lines = [item["words"] for item in resp.get("words_result", [])]
                results[img_path.name] = lines
                print(f"OK ({len(lines)} lines)")
            except Exception as e:
                results[img_path.name] = f"ERROR: {e}"
                print(f"FAILED: {e}")
    
    return results

print("Batch OCR function ready.")
print("Usage: results = batch_ocr(token, '../data/raw/images/')")

## 结果与下一步

### 观察
- 通用文字识别对印刷体效果好，对手写/模糊文字可能有误
- 表格识别能保留表格结构，但复杂表格可能需要后处理
- 高精度版返回位置信息，可用于版面分析和区域裁剪

### 下一步
1. 用真实考古报告图片测试各API效果
2. 对比不同API的识别准确率
3. 设计后处理流程：OCR结果 → 结构化CSV
4. 集成到PDF提取管道中